In [72]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [73]:
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Data Loading and Inspection

In [74]:
df_gym= pd.read_csv('data/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df_fit = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')
df_excise= pd.read_csv('data/3. Fitness Exercises Dataset/exercises_flat.csv')
df_rec = pd.read_excel('data/4. Mendeley Gym Recommendation Dataset/gym recommendation.xlsx')

print("Columns in gym_members_exercise_tracking.csv:", df_gym.columns.tolist())
print("Columns in exercise_dataset.csv:", df_fit.columns.tolist())
print("Columns in exercises_flat.csv:", df_excise.columns.tolist())
print("Columns in gym recommendation.xlsx:", df_rec.columns.tolist())
print()
print(f'Dataset 1 (Gym Members)          : {df_gym.shape[0]:,} rows × {df_gym.shape[1]} cols')
print(f'Dataset 2 (Exercise & Fitness)   : {df_fit.shape[0]:,} rows × {df_fit.shape[1]} cols')
print(f'Dataset 3 (Exercise Library)     : {df_excise.shape[0]:,} rows × {df_excise.shape[1]} cols')
print(f'Dataset 4 (Mendeley Coaching)    : {df_rec.shape[0]:,} rows × {df_rec.shape[1]} cols')
print()

Columns in gym_members_exercise_tracking.csv: ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI']
Columns in exercise_dataset.csv: ['ID', 'Exercise', 'Calories Burn', 'Dream Weight', 'Actual Weight', 'Age', 'Gender', 'Duration', 'Heart Rate', 'BMI', 'Weather Conditions', 'Exercise Intensity']
Columns in exercises_flat.csv: ['exerciseId', 'name', 'gifUrl', 'target_muscles', 'body_part_category', 'equipment_needed', 'secondaryMuscles', 'instructions']
Columns in gym recommendation.xlsx: ['ID', 'Sex', 'Age', 'Height', 'Weight', 'Hypertension', 'Diabetes', 'BMI', 'Level', 'Fitness Goal', 'Fitness Type', 'Exercises', 'Equipment', 'Diet', 'Recommendation']

Dataset 1 (Gym Members)          : 973 rows × 15 cols
Dataset 2 (Exercise & Fitness)   : 3,864 rows × 12 cols
Dataset 3 (Exercise Library)    

In [75]:
print(df_gym.head(3).to_string())
print(df_fit.head(3).to_string())
print(df_excise.head(3).to_string())
print(df_rec.head(3).to_string())


   Age  Gender  Weight (kg)  Height (m)  Max_BPM  Avg_BPM  Resting_BPM  Session_Duration (hours)  Calories_Burned Workout_Type  Fat_Percentage  Water_Intake (liters)  Workout_Frequency (days/week)  Experience_Level    BMI
0   56    Male         88.3        1.71      180      157           60                      1.69           1313.0         Yoga            12.6                    3.5                              4                 3  30.20
1   46  Female         74.9        1.53      179      151           66                      1.30            883.0         HIIT            33.9                    2.1                              4                 2  32.00
2   32  Female         68.1        1.66      167      122           54                      1.11            677.0       Cardio            33.4                    2.3                              4                 2  24.71
   ID    Exercise  Calories Burn  Dream Weight  Actual Weight  Age Gender  Duration  Heart Rate        BMI Weath

In [76]:
for label, df in [("D1 Gym Members", df_gym), ("D2 Fitness", df_fit),("D3 Exercises", df_excise), ("D4 Recommendation", df_rec)]:
    missing = df.isnull().sum()
    has_missing = missing[missing > 0]
    if len(has_missing):
        print(f"\n{label}:\n{has_missing}")
    else:
        print(f"{label}: No missing values")

D1 Gym Members: No missing values
D2 Fitness: No missing values
D3 Exercises: No missing values
D4 Recommendation: No missing values


In [77]:
print("\nDataset 1 (Gym Members) Descriptive Stats")
print(df_gym.describe().round(2).to_string())
 
print("\nDataset 2 (Fitness) Descriptive Stats")
print(df_fit.describe().round(2).to_string())

print("\nDataset 3 (Exercises) Descriptive Stats")
print(df_excise.describe().round(2).to_string())

print("\nDataset 4 (Recommendation) Descriptive Stats")
print(df_rec.describe().round(2).to_string())


Dataset 1 (Gym Members) Descriptive Stats
          Age  Weight (kg)  Height (m)  Max_BPM  Avg_BPM  Resting_BPM  Session_Duration (hours)  Calories_Burned  Fat_Percentage  Water_Intake (liters)  Workout_Frequency (days/week)  Experience_Level     BMI
count  973.00       973.00      973.00   973.00   973.00       973.00                    973.00           973.00          973.00                 973.00                         973.00            973.00  973.00
mean    38.68        73.85        1.72   179.88   143.77        62.22                      1.26           905.42           24.98                   2.63                           3.32              1.81   24.91
std     12.18        21.21        0.13    11.53    14.35         7.33                      0.34           272.64            6.26                   0.60                           0.91              0.74    6.66
min     18.00        40.00        1.50   160.00   120.00        50.00                      0.50           303.00         

In [78]:
print("Dataset 1 (Gym Members) Info:")
df_gym.info()

print("\nDataset 2 (Fitness) Info:")
df_fit.info()

print("\nDataset 3 (Exercises) Info:")
df_excise.info()

print("\nDataset 4 (Recommendation) Info:")
df_rec.info()

Dataset 1 (Gym Members) Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    object 
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    object 
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (d

In [79]:
df_gym.head(3)

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71


In [80]:
df_gym.head(3)

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71


In [81]:
df_excise.head(3)

,exerciseId,name,gifUrl,target_muscles,body_part_category,equipment_needed,secondaryMuscles,instructions
0,2ORFMoR,hack calf raise,2ORFMoR.gif,calves,lower legs,sled machine,hamstrings; glutes,Step:1 Adjust the sled machine to a comfortabl...
1,2Qh2J1e,sled 45Â° leg press (side pov),2Qh2J1e.gif,glutes,upper legs,sled machine,quadriceps; hamstrings; calves,Step:1 Adjust the seat of the sled machine so ...
2,3eGE2JC,dumbbell front raise,3eGE2JC.gif,delts,shoulders,dumbbell,biceps; trapezius,Step:1 Stand with your feet shoulder-width apa...


In [82]:
df_rec.head(3)

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment,Diet,Recommendation
0,1,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,2,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,3,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...


# Data Preprocessing

In [83]:
df_fit_clean = df_fit.copy()

df_fit_clean.drop(columns=['ID', 'Exercise'], inplace=True)

df_fit_clean.rename(columns={
    'Calories Burn' : 'Calories_Burned',
    'Actual Weight' : 'Weight (kg)',
    'Dream Weight' : 'Dream_Weight',
    'Heart Rate' : 'Avg_BPM',
    'Weather Conditions' : 'Weather_Conditions',
    'Exercise Intensity' : 'Exercise_Intensity',
}, inplace=True)

df_fit_clean.rename(columns={'Duration': 'Session_Duration (hours)'}, inplace=True)
df_fit_clean['Session_Duration (hours)'] = df_fit_clean['Session_Duration (hours)'] / 60

#Adding new column name _source to dataset 1 and 2 to identify the source of data
df_gym['_source']      = 'ds1'
df_fit_clean['_source'] = 'ds2'

In [84]:
print('DS1 columns:', list(df_gym.columns))
print()
print('DS2 columns After CLeaning and Renaming:', list(df_fit_clean.columns))

DS1 columns: ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI', '_source']

DS2 columns After CLeaning and Renaming: ['Calories_Burned', 'Dream_Weight', 'Weight (kg)', 'Age', 'Gender', 'Session_Duration (hours)', 'Avg_BPM', 'BMI', 'Weather_Conditions', 'Exercise_Intensity', '_source']


In [85]:
# pd.concat fills missing columns with NaN automatically
dfUnified = pd.concat([df_gym, df_fit_clean], ignore_index=True)

print(f'Merged DataFrame shape: {dfUnified.shape}')
print(f'  DS1 (gym data) rows: {(dfUnified["_source"]=="ds1").sum()}')
print(f'  DS2 (fit data) rows: {(dfUnified["_source"]=="ds2").sum()}')
print()
print('Missing values per column:')
print(dfUnified.isnull().sum().to_string())

Merged DataFrame shape: (4837, 19)
  DS1 (gym data) rows: 973
  DS2 (fit data) rows: 3864

Missing values per column:
Age                                 0
Gender                              0
Weight (kg)                         0
Height (m)                       3864
Max_BPM                          3864
Avg_BPM                             0
Resting_BPM                      3864
Session_Duration (hours)            0
Calories_Burned                     0
Workout_Type                     3864
Fat_Percentage                   3864
Water_Intake (liters)            3864
Workout_Frequency (days/week)    3864
Experience_Level                 3864
BMI                                 0
_source                             0
Dream_Weight                      973
Weather_Conditions                973
Exercise_Intensity                973


In [86]:
dfUnified.head(3)

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI,_source,Dream_Weight,Weather_Conditions,Exercise_Intensity
0,56,Male,88.3,1.71,180.0,157,60.0,1.69,1313.0,Yoga,12.6,3.5,4.0,3.0,30.20,ds1,NaN,NaN,NaN
1,46,Female,74.9,1.53,179.0,151,66.0,1.30,883.0,HIIT,33.9,2.1,4.0,2.0,32.00,ds1,NaN,NaN,NaN
2,32,Female,68.1,1.66,167.0,122,54.0,1.11,677.0,Cardio,33.4,2.3,4.0,2.0,24.71,ds1,NaN,NaN,NaN


In [87]:
numeric_cols = dfUnified.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Calories_Burned']

for col in numeric_cols:
    if dfUnified[col].isnull().sum() > 0:
        median_val = dfUnified[col].median()
        dfUnified[col].fillna(median_val, inplace=True)
        print(f'  Filled "{col}" NaN with median={median_val:.2f}')
print(dfUnified.isnull().sum().to_string())

  Filled "Height (m)" NaN with median=1.71
  Filled "Max_BPM" NaN with median=180.00
  Filled "Resting_BPM" NaN with median=62.00
  Filled "Fat_Percentage" NaN with median=26.20
  Filled "Water_Intake (liters)" NaN with median=2.60
  Filled "Workout_Frequency (days/week)" NaN with median=3.00
  Filled "Experience_Level" NaN with median=2.00
  Filled "Dream_Weight" NaN with median=75.52
  Filled "Exercise_Intensity" NaN with median=5.00
Age                                 0
Gender                              0
Weight (kg)                         0
Height (m)                          0
Max_BPM                             0
Avg_BPM                             0
Resting_BPM                         0
Session_Duration (hours)            0
Calories_Burned                     0
Workout_Type                     3864
Fat_Percentage                      0
Water_Intake (liters)               0
Workout_Frequency (days/week)       0
Experience_Level                    0
BMI                         

In [88]:
for col in ['Workout_Type', 'Weather_Conditions']:
    if col in dfUnified.columns and dfUnified[col].isnull().sum() > 0:
        mode_val = dfUnified[col].mode()[0]
        dfUnified[col].fillna(mode_val, inplace=True)
        print(f'  Filled "{col}" NaNs with mode="{mode_val}"')

print(dfUnified.isnull().sum().to_string())

  Filled "Workout_Type" NaNs with mode="Strength"
  Filled "Weather_Conditions" NaNs with mode="Cloudy"
Age                              0
Gender                           0
Weight (kg)                      0
Height (m)                       0
Max_BPM                          0
Avg_BPM                          0
Resting_BPM                      0
Session_Duration (hours)         0
Calories_Burned                  0
Workout_Type                     0
Fat_Percentage                   0
Water_Intake (liters)            0
Workout_Frequency (days/week)    0
Experience_Level                 0
BMI                              0
_source                          0
Dream_Weight                     0
Weather_Conditions               0
Exercise_Intensity               0


In [89]:
dfUnified.drop(columns=['_source'], inplace=True)
print('Removed "_source" column. Current columns:', list(dfUnified.columns))

print(f'\nRechecking NaNs: {dfUnified.isnull().sum().sum()}')

Removed "_source" column. Current columns: ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI', 'Dream_Weight', 'Weather_Conditions', 'Exercise_Intensity']

Rechecking NaNs: 0


In [90]:
beforeLength = len(dfUnified)
dfUnified.drop_duplicates(inplace=True)
print('Duplicates removed:', beforeLength - len(dfUnified))

Duplicates removed: 0


In [92]:
Q1 = dfUnified['Calories_Burned'].quantile(0.25)
Q3 = dfUnified['Calories_Burned'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

In [93]:
afterLength = len(dfUnified)
dfUnified = dfUnified[(dfUnified['Calories_Burned'] >= lower) & (dfUnified['Calories_Burned'] <= upper)]
print(f'Outliers removed: {afterLength - len(dfUnified)}')

Outliers removed: 136


In [94]:
print('Final Shape:', dfUnified.shape)

Final Shape: (4701, 18)


In [95]:
dfUnified['Gender'] = dfUnified['Gender'].map({'Male': 1, 'Female': 0}) #Male: 1, Female: 0

dfUnified = pd.get_dummies(dfUnified, columns=['Workout_Type'], prefix='wt', drop_first=False) #encoding for Workout_Type: wt_Cardio, wt_Strength, wt_Flexibility

weather_map = {'Sunny': 2, 'Cloudy': 1, 'Rainy': 0}
dfUnified['Weather_Conditions'] = dfUnified['Weather_Conditions'].map(weather_map)
dfUnified['Weather_Conditions'].fillna(dfUnified['Weather_Conditions'].median(), inplace=True) #Sunny=2, Cloudy=1, Rainy=0

bool_cols = dfUnified.select_dtypes(include='bool').columns
dfUnified[bool_cols] = dfUnified[bool_cols].astype(int) #convert bool into int (True=1, False=0) for wt_Cardio, wt_Strength, wt_Flexibility

In [99]:
print('Encoded columns:', [c for c in dfUnified.columns if c.startswith('wt_') or c in ['Gender','Weather_Conditions']])
print(f'DataFrame shape after encoding: {dfUnified.shape}')
dfUnified.head(3)

Encoded columns: ['Gender', 'Weather_Conditions', 'wt_Cardio', 'wt_HIIT', 'wt_Strength', 'wt_Yoga']
DataFrame shape after encoding: (4701, 21)


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Fat_Percentage,...,Workout_Frequency (days/week),Experience_Level,BMI,Dream_Weight,Weather_Conditions,Exercise_Intensity,wt_Cardio,wt_HIIT,wt_Strength,wt_Yoga
1,46,0,74.9,1.53,179.0,151,66.0,1.30,883.0,33.9,...,4.0,2.0,32.00,75.522136,1,5.0,0,1,0,0
2,32,0,68.1,1.66,167.0,122,54.0,1.11,677.0,33.4,...,4.0,2.0,24.71,75.522136,1,5.0,1,0,0,0
3,25,1,53.2,1.70,190.0,164,56.0,0.59,532.0,28.8,...,3.0,1.0,18.41,75.522136,1,5.0,0,0,1,0
